# 🧠 Smart Waste Hub - Autonomous Model Trainer
This notebook will connect to your **MongoDB Atlas** database, download the user-provided images and labels, and retrain the waste classification model using **Transfer Learning** (MobileNetV2).

### 🚀 How to use:
1.  **Enter your MONGO_URI** in the first cell.
2.  **Run All Cells.**
3.  **Download the result:** You will get a `model.json` and weight files to upload to your project's `public/tfjs_model` folder.

In [ ]:
!pip install pymongo dnspython tensorflowjs tensorflow pillow

import os
import numpy as np
import requests
from pymongo import MongoClient
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import base64
from PIL import Image
import io

# ── 🔑 CONFIGURATION ──────────────────────────────────────────────────────────
MONGO_URI = "REPLACE_WITH_YOUR_MONGO_URI" # Enter your string here!
DB_NAME = "smart-waste-hub"
COLLECTION_NAME = "datasets"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

### 📥 Step 1: Download Data from MongoDB

In [ ]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

print(f"📦 Found {collection.count_documents({})} learning samples. Preparing dataset...")

# Create directories for training
os.makedirs('dataset/GREEN_BIN', exist_ok=True)
os.makedirs('dataset/BLUE_BIN', exist_ok=True)
os.makedirs('dataset/RED_BIN', exist_ok=True)

for i, doc in enumerate(collection.find()):
    try:
        label = doc['label']
        # Decode base64 image
        image_data = doc['imageData'].split(",")[1]
        image = Image.open(io.BytesIO(base64.b64decode(image_data)))
        image = image.convert("RGB").resize(IMAGE_SIZE)
        image.save(f'dataset/{label}/{i}.jpg')
    except Exception as e:
        print(f"⚠️ Error processing sample {i}: {e}")

print("✅ Dataset preparation complete!")

### 🏗️ Step 2: Build & Train Model (Transfer Learning)

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, horizontal_flip=True, rotation_range=20)

train_generator = train_datagen.flow_from_directory(
    'dataset', target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='training')

base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical-crossentropy', metrics=['accuracy'])
model.fit(train_generator, epochs=5)
print("✅ Model training finished!")

### 📤 Step 3: Export to TensorFlow.js

In [ ]:
import tensorflowjs as tfjs
tfjs.converters.save_keras_model(model, 'tfjs_model')
print("🚀 Success! Your new model is ready in the 'tfjs_model' folder. Download it and upload to your project!")